In [3]:
!pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 14.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.3
    Uninstalling scipy-1.15.3:
      Successfully uninstalled scipy-1.15.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatib

In [1]:
import gensim
from gensim.models import Word2Vec,KeyedVectors
import re
import nltk

In [2]:
import gensim.downloader as api

wv = api.load('word2vec-google-news-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [3]:
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [4]:
import pandas as pd

messages = pd.read_csv("/content/SMSSpamCollection.txt",sep='\t',names=['label','message'])


In [10]:
messages.shape

(5572, 2)

Lemmatization

In [5]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()


In [6]:
corpus = []

for i in range(0,len(messages)):
  review = re.sub('[^a-zA-Z]',' ',messages['message'][i])
  review = review.lower().split()
  review = [lemmatizer.lemmatize(word) for word in review]
  review = ' '.join(review)
  corpus.append(review)

corpus


['go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat',
 'ok lar joking wif u oni',
 'free entry in a wkly comp to win fa cup final tkts st may text fa to to receive entry question std txt rate t c s apply over s',
 'u dun say so early hor u c already then say',
 'nah i don t think he go to usf he life around here though',
 'freemsg hey there darling it s been week s now and no word back i d like some fun you up for it still tb ok xxx std chgs to send to rcv',
 'even my brother is not like to speak with me they treat me like aid patent',
 'a per your request melle melle oru minnaminunginte nurungu vettam ha been set a your callertune for all caller press to copy your friend callertune',
 'winner a a valued network customer you have been selected to receivea prize reward to claim call claim code kl valid hour only',
 'had your mobile month or more u r entitled to update to the latest colour mobile with camera for free call the mobile up

In [9]:
[[i,j,k] for i,j,k in zip(list(map(len,corpus)),corpus,messages['message']) if i<1]
#checking for messages that are empty

[[0, '', '645'], [0, '', ':) '], [0, '', ':-) :-)']]

So 3 messages are empty: 5572-3 = 5569 messages remaining

In [11]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess    #simple_preprocess converts a doc into a list of lowercase tokens

In [12]:
words = []
for sent in corpus:
  sent_token = sent_tokenize(sent)    #converting sentence to words
  for sent in sent_token:
    words.append(simple_preprocess(sent))   #converting each word of every sentence into lowercase and then appending lists of words of each sent in words


In [13]:
words

[['go',
  'until',
  'jurong',
  'point',
  'crazy',
  'available',
  'only',
  'in',
  'bugis',
  'great',
  'world',
  'la',
  'buffet',
  'cine',
  'there',
  'got',
  'amore',
  'wat'],
 ['ok', 'lar', 'joking', 'wif', 'oni'],
 ['free',
  'entry',
  'in',
  'wkly',
  'comp',
  'to',
  'win',
  'fa',
  'cup',
  'final',
  'tkts',
  'st',
  'may',
  'text',
  'fa',
  'to',
  'to',
  'receive',
  'entry',
  'question',
  'std',
  'txt',
  'rate',
  'apply',
  'over'],
 ['dun', 'say', 'so', 'early', 'hor', 'already', 'then', 'say'],
 ['nah',
  'don',
  'think',
  'he',
  'go',
  'to',
  'usf',
  'he',
  'life',
  'around',
  'here',
  'though'],
 ['freemsg',
  'hey',
  'there',
  'darling',
  'it',
  'been',
  'week',
  'now',
  'and',
  'no',
  'word',
  'back',
  'like',
  'some',
  'fun',
  'you',
  'up',
  'for',
  'it',
  'still',
  'tb',
  'ok',
  'xxx',
  'std',
  'chgs',
  'to',
  'send',
  'to',
  'rcv'],
 ['even',
  'my',
  'brother',
  'is',
  'not',
  'like',
  'to',
  'spea

In [14]:
#train Word2Vec model on the words list from scratch
model = gensim.models.Word2Vec(words)   #here vector_size by default is 100

model.wv.index_to_key     #get all the vocabulary

['to',
 'you',
 'the',
 'it',
 'and',
 'in',
 'is',
 'me',
 'my',
 'for',
 'your',
 'call',
 'of',
 'that',
 'have',
 'on',
 'now',
 'are',
 'can',
 'so',
 'but',
 'not',
 'or',
 'we',
 'do',
 'get',
 'at',
 'ur',
 'will',
 'if',
 'be',
 'with',
 'no',
 'just',
 'this',
 'gt',
 'lt',
 'go',
 'how',
 'up',
 'when',
 'ok',
 'day',
 'what',
 'free',
 'from',
 'all',
 'out',
 'know',
 'll',
 'come',
 'like',
 'good',
 'time',
 'am',
 'then',
 'got',
 'wa',
 'there',
 'he',
 'love',
 'text',
 'only',
 'want',
 'send',
 'one',
 'need',
 'txt',
 'today',
 'by',
 'going',
 'don',
 'stop',
 'home',
 'she',
 'about',
 'lor',
 'sorry',
 'see',
 'still',
 'mobile',
 'take',
 'back',
 'da',
 'reply',
 'dont',
 'our',
 'think',
 'tell',
 'week',
 'hi',
 'phone',
 'they',
 'new',
 'please',
 'later',
 'pls',
 'any',
 'her',
 'ha',
 'co',
 'did',
 'been',
 'msg',
 'min',
 'some',
 'an',
 'night',
 'make',
 'dear',
 'who',
 'here',
 'message',
 'say',
 'well',
 'where',
 're',
 'thing',
 'much',
 'oh',

In [15]:
model.corpus_count

5569

In [16]:
model.epochs

5

An epoch refers to one complete pass through the entire training dataset.Here by default when model was created epoch count was 5

In [17]:
model.wv.similar_by_word('good')

[('morning', 0.9987377524375916),
 ('well', 0.9987229704856873),
 ('night', 0.9986638426780701),
 ('my', 0.9986199140548706),
 ('all', 0.9985952973365784),
 ('oh', 0.9985676407814026),
 ('happy', 0.9985572695732117),
 ('wa', 0.9985566139221191),
 ('day', 0.9984786510467529),
 ('got', 0.9984530210494995)]

The similarity is typically measured by the cosine similarity between the word vectors. The output is a list of tuples, where each tuple contains a similar word to 'good' from the trained words wv and its similarity score.

In [18]:
model.wv['good'].shape

(100,)

**AvgWord2Vec**

With this we can get a single vector of 300 dim. for one sentence by finding the average row wise of all the vectors of all words in a sentence.

In [20]:
model.vector_size

100

In [21]:
np.zeros(100)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [19]:
import numpy as np

def avgword2vec(doc):
  vec_list = [model.wv[word] for word in doc if word in model.wv.index_to_key]
  if not vec_list:  # Check if the list of vectors is empty
    # If empty, return a zero vector of the correct dimensionality
    return np.zeros(model.vector_size)
  else:
    return np.mean(vec_list, axis=0)     #row wise mean of all vectors of the words

In [14]:
!pip install tqdm

In [22]:
from tqdm import tqdm
#used to visualize progress bars

In [28]:
len(words)

5569

In [23]:
#apply for entire sentences

X = []
for i in tqdm(range(len(words))):
  X.append(avgword2vec(words[i]))     #averaged vectors are appended into X sent by sentence

100%|██████████| 5569/5569 [00:00<00:00, 8830.66it/s]


In [39]:
X

[array([-0.2174785 ,  0.19619897,  0.00076735,  0.12267397,  0.16747414,
        -0.51677316,  0.1511263 ,  0.5313377 , -0.32376918, -0.10065555,
        -0.20288375, -0.40455976, -0.11802299,  0.12218717,  0.13695286,
        -0.16580644,  0.12883164, -0.28363845, -0.10797337, -0.5276107 ,
         0.1764511 ,  0.1541848 ,  0.14634366, -0.18748635, -0.06159757,
        -0.04320003, -0.17321378, -0.20157464, -0.35714167,  0.07131469,
         0.31496102, -0.01852621,  0.07464977, -0.21911673, -0.06194558,
         0.3975673 ,  0.0195502 , -0.14629403, -0.12555356, -0.4077094 ,
         0.07914186, -0.31804675, -0.1266658 ,  0.0053434 ,  0.18837687,
        -0.0290714 , -0.09228606, -0.13541296,  0.18745445,  0.25982583,
         0.18895051, -0.24359156, -0.04121409,  0.02674617, -0.14840978,
         0.21456832,  0.21302351,  0.12587966, -0.30241558,  0.19521645,
         0.08595773,  0.09616645, -0.04942845, -0.04479546, -0.3207417 ,
         0.20907731,  0.04599024,  0.17650224, -0.3

In [24]:
len(X)

5569

In [26]:
#independent features
X_new = np.array(X)

In [25]:
X_new

array([0.00393227, 0.00314144, 0.00197173, ..., 0.00731287, 0.00537668,
       0.00573873])

In [27]:
X_new.shape

(5569, 100)

In [45]:
X_new[0]

array([-0.2174785 ,  0.19619897,  0.00076735,  0.12267397,  0.16747414,
       -0.51677316,  0.1511263 ,  0.53133768, -0.32376918, -0.10065555,
       -0.20288375, -0.40455976, -0.11802299,  0.12218717,  0.13695286,
       -0.16580644,  0.12883164, -0.28363845, -0.10797337, -0.52761072,
        0.1764511 ,  0.1541848 ,  0.14634366, -0.18748635, -0.06159757,
       -0.04320003, -0.17321378, -0.20157464, -0.35714167,  0.07131469,
        0.31496102, -0.01852621,  0.07464977, -0.21911673, -0.06194558,
        0.3975673 ,  0.0195502 , -0.14629403, -0.12555356, -0.40770939,
        0.07914186, -0.31804675, -0.1266658 ,  0.0053434 ,  0.18837687,
       -0.0290714 , -0.09228606, -0.13541296,  0.18745445,  0.25982583,
        0.18895051, -0.24359156, -0.04121409,  0.02674617, -0.14840978,
        0.21456832,  0.21302351,  0.12587966, -0.30241558,  0.19521645,
        0.08595773,  0.09616645, -0.04942845, -0.04479546, -0.32074171,
        0.20907731,  0.04599024,  0.17650224, -0.33615035,  0.32

In [46]:
X_new[0].shape

(100,)

In [26]:
import numpy as np

# Set print options to display the full array
np.set_printoptions(threshold=np.inf)

# Display the entire array
print(X_new)

# Reset print options to default (optional, but good practice)
np.set_printoptions(threshold=1000)

[ 0.00393227  0.00314144  0.00197173  0.00573838  0.00565768  0.00574581
  0.00620124  0.00413873  0.00343688  0.00305357  0.00588982  0.00293319
  0.00356258  0.00669832  0.00555981  0.0034198   0.00613418  0.00435632
  0.00794487  0.00222286  0.00790051  0.00537024  0.00464138  0.00325702
  0.00753084  0.0042837   0.00582831  0.00853912  0.00804142  0.00735497
  0.00738644  0.00755222  0.00880684  0.00800873  0.00504837  0.00486286
  0.00748401  0.00612746  0.00308861  0.00700671  0.0055939   0.00878615
  0.00332208  0.00985196  0.01302206  0.00180034  0.00721008  0.0058089
  0.00600257  0.00589932  0.00879411  0.01209157  0.00539389  0.00751675
  0.00361291  0.01289185  0.00379645  0.0040139   0.00812454  0.00614509
  0.00562214  0.00748283  0.00542844  0.00531096  0.00405004  0.00547847
  0.00481773  0.00158754  0.00786341  0.00449325  0.00401176  0.00956069
  0.0062571   0.01320221  0.00412237  0.00459694  0.00688312  0.00812007
  0.01417344  0.00733058  0.00341289  0.00551725  0.

In [68]:
#Dependent Feature

y = messages[list(map(lambda x: len(x)>0, corpus))]   #storing the non-empty messages only
y = pd.get_dummies(y['label'])

In [65]:
y

,ham,spam
0,True,False
1,True,False
2,False,True
3,True,False
4,True,False
...,...,...
5567,False,True
5568,True,False
5569,True,False
5570,True,False


In [69]:
y = y.iloc[:,0].values.astype(int)    #taking only ham column and convert them into int
y

array([1, 1, 0, ..., 1, 1, 1])

In [70]:
y.shape

(5569,)

In [71]:
#Converting X into Dataframe

df = pd.DataFrame(X_new)

In [72]:
X=df

In [73]:
X.shape

(5569, 100)

In [74]:
X.head()

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.141282,0.203986,0.030557,0.042754,0.076076,-0.357567,0.166287,0.635811,-0.160615,-0.168501,...,0.321117,0.105557,0.041486,0.057558,0.550967,0.252567,0.190983,-0.213911,0.184157,-0.046509
1,-0.133930,0.174860,0.024386,0.037807,0.072067,-0.308199,0.131988,0.550221,-0.138623,-0.140861,...,0.283392,0.084797,0.031041,0.044376,0.464202,0.210919,0.165038,-0.194981,0.164018,-0.045726
2,-0.156342,0.218435,0.037191,0.059861,0.055148,-0.395766,0.169768,0.633565,-0.170392,-0.195826,...,0.320763,0.102519,0.024169,0.042381,0.578264,0.241379,0.152176,-0.257559,0.214094,-0.035338
3,-0.196017,0.283290,0.036110,0.061718,0.104286,-0.488806,0.225440,0.883079,-0.222792,-0.227919,...,0.442577,0.145132,0.061756,0.086668,0.748962,0.351470,0.280569,-0.295067,0.251546,-0.073511
4,-0.171911,0.227239,0.035887,0.050546,0.094985,-0.408785,0.183881,0.737238,-0.188911,-0.198339,...,0.372015,0.117000,0.053093,0.072634,0.621223,0.290400,0.223144,-0.257605,0.204491,-0.059122


In [80]:
X.isnull().sum()

,0
0,0
1,0
2,0
3,0
4,0
...,...
95,0
96,0
97,0
98,0


In [75]:
#train test split

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)

In [76]:
X_train.shape,X_test.shape

((4455, 100), (1114, 100))

In [77]:
y_train

array([1, 0, 1, ..., 1, 1, 1])

Model Training

In [78]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier()

In [82]:
classifier.fit(X_train,y_train)

RandomForestClassifier()

In [83]:
y_pred = classifier.predict(X_test)

In [84]:
#Performance metrics

from sklearn.metrics import accuracy_score,classification_report
print(accuracy_score(y_test,y_pred))

0.9649910233393177


In [85]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.86      0.86      0.86       141
           1       0.98      0.98      0.98       973

    accuracy                           0.96      1114
   macro avg       0.92      0.92      0.92      1114
weighted avg       0.96      0.96      0.96      1114

